<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L01-first-contact/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/flagships/humanoid-lab/lessons/F15-L01-first-contact/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=flagships/humanoid-lab/lessons/F15-L01-first-contact/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have. The cell below installs `mujoco==3.13.0` and fetches the
files it needs beside it, and does nothing where they are already present. On Kaggle, switch
Internet on in the notebook's settings first; Kaggle allows that only for phone-verified
accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("mujoco", "mujoco==3.13.0")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/humanoid.xml"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/flagships/humanoid-lab/lessons/F15-L01-first-contact/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# F15-L01 · First contact with a humanoid

**You will build:** a three-instrument panel for MuJoCo's own humanoid — a clock
(`step_for`), a posture sensor (`com_height`) and a stopwatch (`real_time_factor`).

**Time:** ~45 minutes · **Runs on:** a laptop CPU, no GPU, no download
· **Prerequisites:** none

By the end you will be able to:
1. Locate any simulation quantity in either `mjModel` or `mjData`, and say why it lives there.
2. Implement `step_for(seconds)` so that simulated time advances by a whole number of steps.
3. Compute whole-body centre-of-mass height from state, and show it differs from root height.
4. Measure this machine's real-time factor and explain why simulated time is not wall time.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import math
import sys
import time
import urllib.request
from pathlib import Path
from typing import Callable

import mujoco
import numpy as np

print("mujoco", mujoco.__version__, "· numpy", np.__version__)

# The one model we use, shipped next to this notebook in assets/ (Apache-2.0, see SOURCE.md).
MODEL_URL = (
    "https://raw.githubusercontent.com/google-deepmind/mujoco/main/model/humanoid/humanoid.xml"
)
MODEL_FILENAME = "humanoid.xml"

# Simulated time accumulates in floating point, so an exact comparison against a target is
# unreliable by a few parts in 1e14. Every time comparison in this lesson uses this slack.
TIME_EPS = 1e-9


def humanoid_xml_path() -> Path:
    """Return the path to the cached humanoid XML, downloading it only if it is missing.

    The file ships inside this lesson's assets/ directory, so the normal path is offline and
    nothing is fetched. The download branch exists only for a truncated checkout.
    """
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    candidates = [
        here / "assets" / MODEL_FILENAME,
        here.parent / "assets" / MODEL_FILENAME,
        Path.cwd() / "assets" / MODEL_FILENAME,
        Path.cwd().parent / "assets" / MODEL_FILENAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    target = candidates[0]
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f"cached model absent; fetching once from {MODEL_URL}")
    urllib.request.urlretrieve(MODEL_URL, target)  # noqa: S310 - pinned https URL above
    return target


def load_humanoid():
    """Compile the humanoid XML and hand back a fresh (model, data) pair."""
    model = mujoco.MjModel.from_xml_path(str(humanoid_xml_path()))
    return model, mujoco.MjData(model)


MODEL, DATA = load_humanoid()
print("compiled:", humanoid_xml_path().name)

# Run all is safe before you have written a line: every check, and every demo that needs your
# code, goes through _try, which reports and carries on. The board at the foot of the notebook
# shows where you stand.
_EXERCISES = {"exercise 1": "step_for", "exercise 2": "com_height",
              "exercise 3": "real_time_factor", "self-check": "your four letters"}
_STATUS: dict = {}      # label -> "passed" | "failed" | "not started", read by the board


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. Anything listed in `needs` must have
    passed first; until it has, this names it and skips rather than failing on its behalf.
    Nothing is swallowed: every outcome lands in _STATUS, and the `__main__` block at the foot
    of this file exits non-zero outside a notebook if any check failed.
    """
    waiting = [n for n in needs if _STATUS.get(n) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        named = " and ".join(f"{n} ({_EXERCISES.get(n, n)})" for n in waiting)
        one = len(waiting) == 1
        print(f"{label}: skipped until {named} {'passes' if one else 'pass'} — finish "
              f"{'that' if one else 'those'}, then re-run this cell.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        print(f"{label}: {exc}" if str(exc) else
              f"{label}: not implemented yet — fill in the stub above, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Two objects, and the difference is the whole lesson

`mjModel` is the compiled description: what the robot *is*. It does not change while you
simulate. `mjData` is the state: what the robot is *doing right now* — positions,
velocities, forces, the clock. One model can serve many independent `mjData` instances.

Run the cell. Every number below is read out of the compiled model, not typed by hand.

In [ ]:
print("mjModel — constant description")
for _name in ("nq", "nv", "nu", "nbody", "njnt", "ngeom", "nkey"):
    print(f"  {_name:7s} = {getattr(MODEL, _name)}")
print(f"  timestep = {MODEL.opt.timestep} s")
print(f"  gravity  = {MODEL.opt.gravity[2]:.2f} m/s^2 (z)")
print(f"  mass     = {MODEL.body_mass.sum():.3f} kg total")

print("\nmjData — live state")
print(f"  time = {DATA.time} s")
print(f"  qpos holds {DATA.qpos.shape[0]} numbers, qvel holds {DATA.qvel.shape[0]}")

In [ ]:
# Two scratch pads, one model: mjData instances are independent.
_left, _right = mujoco.MjData(MODEL), mujoco.MjData(MODEL)
for _ in range(10):
    mujoco.mj_step(MODEL, _left)
print(f"left.time={_left.time:.3f}s  right.time={_right.time:.3f}s  "
      f"(the model they share was never modified)")

## 2. Why `nq` is not `nv`

A free joint carries a 3-vector position and a 4-number quaternion in `qpos`, but only a
linear and an angular 3-vector in `qvel`. Orientation therefore costs one more slot in
`qpos` than in `qvel`. Hinges cost one of each.

Run this to see where each joint's numbers actually live.

In [ ]:
def joint_table(model, limit: int = 6) -> list:
    """Rows of (name, joint type, qpos address, qvel address) for the first `limit` joints."""
    rows = []
    for j in range(min(model.njnt, limit)):
        rows.append((
            mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, j),
            mujoco.mjtJoint(model.jnt_type[j]).name,
            int(model.jnt_qposadr[j]),
            int(model.jnt_dofadr[j]),
        ))
    return rows


print(f"{'joint':18s} {'type':12s} {'qpos@':>6s} {'qvel@':>6s}")
for _jname, _jtype, _qadr, _vadr in joint_table(MODEL):
    print(f"{_jname:18s} {_jtype:12s} {_qadr:6d} {_vadr:6d}")
print(f"\nnq - nv = {MODEL.nq - MODEL.nv}  (one quaternion's worth of extra position slots)")
print(f"free joint qpos block: {np.array(MODEL.qpos0[:7])}")
print("                       ^ x y z, then quaternion w x y z")

## 3. One step of physics

`mujoco.mj_step(model, data)` integrates the state forward by exactly `model.opt.timestep`
seconds of *simulated* time and advances `data.time` by the same amount. It is the only
thing that moves the clock. Anything you assign to `data.time` simulates nothing.

In [ ]:
_probe = mujoco.MjData(MODEL)
_t_before, _z_before = _probe.time, float(_probe.qpos[2])
mujoco.mj_step(MODEL, _probe)
print(f"time   {_t_before} -> {_probe.time}  (+{_probe.time - _t_before} s)")
print(f"root z {_z_before:.6f} -> {float(_probe.qpos[2]):.6f}  (it has begun to fall)")

## 4. Exercise 1 — `step_for(model, data, seconds)`

Simulated time only moves in whole timesteps, so a request for an arbitrary number of
seconds cannot be honoured exactly. This lesson's contract is **never undershoot**: keep
stepping until at least `seconds` of simulated time has elapsed, then stop. The answer is
therefore a *ceiling*, not a truncation, and you must find it by stepping and watching
`data.time` — not by assigning to the clock.

<details><summary>💡 Hint 1 — what to think about</summary>

Two questions decide this: what is your stopping test measured *from*, and can your loop run
zero times? The clock need not read zero when you are called — a second call on the same
`data` starts where the first one stopped — and a request for no time at all costs nothing.

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Note the clock on entry. While the time elapsed since then is still short of the request
(less `TIME_EPS`, so float drift cannot buy an extra step), take one real `mujoco.mj_step`
and count it. Return the count. Counting steps as you take them rounds up on its own,
whatever the timestep, so there is nothing to truncate and nothing to hard-code.

</details>

In [ ]:
def step_for(model, data, seconds: float) -> int:
    """Step `data` until at least `seconds` of simulated time has passed; return step count.

    Measure elapsed time from `data.time` as it is when you are called, so the function also
    works on a `data` that has already been stepped. Use `TIME_EPS` as the slack in your
    comparison, so float drift in the accumulated clock cannot buy an extra step.

    Example (this model's timestep is 0.005 s, so 0.007 s is 1.4 timesteps, rounded up to 2):
        >>> model, data = load_humanoid()
        >>> step_for(model, data, 0.007)
        2
        >>> round(data.time, 3)
        0.01

    Returns:
        The number of mujoco.mj_step calls made. Zero is a valid answer for seconds <= 0.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public check — run it as often as you like.
def _check_step_for() -> None:
    model, data = load_humanoid()
    n = step_for(model, data, 1.0)
    expected = math.ceil(1.0 / model.opt.timestep - TIME_EPS)
    assert n == expected, (
        f"step_for(1.0) returned {n}, expected {expected} — divide the request by "
        "model.opt.timestep; do not assume any particular timestep value."
    )
    assert abs(data.time - n * model.opt.timestep) < 1e-6, (
        f"data.time is {data.time} after {n} steps — you counted steps without calling "
        "mujoco.mj_step, so no physics actually happened."
    )

    model2, data2 = load_humanoid()
    partial = step_for(model2, data2, 0.007)
    assert partial == 2, (
        f"step_for(0.007) returned {partial}, expected 2 — int() and // truncate, and the "
        "contract is to never undershoot, so the step count rounds up."
    )

    model3, data3 = load_humanoid()
    assert step_for(model3, data3, 0.0) == 0, (
        "step_for(model, data, 0.0) must take no steps — your loop runs at least once, so "
        "make it a while loop that tests before it steps."
    )

    model4, data4 = load_humanoid()
    first = step_for(model4, data4, 0.5)
    second = step_for(model4, data4, 0.5)
    assert first == second, (
        f"stepping 0.5 s twice gave {first} then {second} — you compared data.time against "
        "`seconds` absolutely instead of against the time when the call started."
    )
    print(f"exercise 1 looks right: {n} steps buy {data.time:.3f} s of simulated time")

In [ ]:
_try("exercise 1", _check_step_for)

## 5. Exercise 2 — `com_height(model, data)`

`data.qpos[2]` is the height of the *root body* — the torso the free joint attaches to. The
whole-body centre of mass is the mass-weighted average over every body: a different number,
and the one that tells you whether a humanoid has fallen.

Two ingredients: `model.body_mass` (constant, so it lives in the model) and `data.xipos`
(each body's centre-of-mass position in world coordinates, so it lives in the data). Note
`xipos`, not `xpos`: `xpos` is the body *frame origin*, which is not where its mass sits.

<details><summary>💡 Hint 1 — what to think about</summary>

Three wrong answers are each one slip away, and the check prints all three for your pose:
the root's height, a weighting of the body frame origins, and a plain average that forgets a
foot weighs less than a torso. Which array says where each body's mass sits, and which
object knows how much each body weighs?

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Take the z component of every row of `data.xipos`, weight each by the matching entry of
`model.body_mass`, add them up and divide by the total mass. Return a plain Python float,
not a one-element array. Do not call `mj_forward` in here; refreshing is the caller's job.

</details>

In [ ]:
def com_height(model, data) -> float:
    """Return the z coordinate of the whole-body centre of mass, in metres.

    Mass-weighted mean over all bodies: sum(mass_i * z_i) / sum(mass_i), where z_i is
    `data.xipos[i][2]`. The world body carries zero mass, so including it changes nothing.

    `data.xipos` is only valid after MuJoCo has run kinematics — after an `mj_step`, an
    `mj_forward`, or a keyframe reset followed by `mj_forward`. This function does not
    refresh it for you; the caller does.

    Example (the default pose of this model stands taller at the root than at the COM):
        >>> model, data = load_humanoid()
        >>> mujoco.mj_forward(model, data)
        >>> com_height(model, data) < float(data.qpos[2])
        True

    Returns:
        A single float, not an array.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_com_height() -> None:
    model, data = load_humanoid()
    mujoco.mj_forward(model, data)
    got = com_height(model, data)
    assert isinstance(got, float), (
        f"com_height returned {type(got).__name__} — index the z component and cast with "
        "float(); do not return the whole 3-vector."
    )
    reference = float(data.subtree_com[0][2])
    mass = model.body_mass
    assert abs(got - reference) < 1e-9, (
        f"com_height gave {got:.6f}, MuJoCo's own whole-body COM is {reference:.6f}. Each "
        "near miss names a different mistake, and all three are measured from this pose: "
        f"{float(data.qpos[2]):.6f} is the root height in qpos[2]; "
        f"{float((mass * data.xpos[:, 2]).sum() / mass.sum()):.6f} weights data.xpos, the "
        "frame origins, where data.xipos belongs; "
        f"{float(data.xipos[:, 2].mean()):.6f} is an unweighted mean of xipos, with "
        "model.body_mass left out."
    )
    squat = mujoco.MjData(model)
    mujoco.mj_resetDataKeyframe(model, squat, 0)
    mujoco.mj_forward(model, squat)
    crouched = com_height(model, squat)
    assert crouched < got - 0.1, (
        f"the crouched keyframe gave {crouched:.6f} against {got:.6f} standing — a constant, "
        "or anything computed from the model alone, cannot respond to a pose change; the "
        "pose lives in data.xipos."
    )
    print(f"exercise 2 looks right: standing COM {got:.3f} m, crouched {crouched:.3f} m")

In [ ]:
_try("exercise 2", _check_com_height)

## 6. Exercise 3 — `real_time_factor(model, data, seconds)`

Simulated time is bookkeeping; wall-clock time is what you actually wait. Their ratio is
the real-time factor: above 1 the simulation outruns reality, below 1 it lags. It is a
property of *this machine plus this model*, so it is measured, never assumed.

Reset `data` with `mujoco.mj_resetData` first so every measurement starts from the same
pose, time the stepping with `time.perf_counter` (not `time.time`, a wall clock subject to
adjustment mid-measurement), and reuse your own `step_for`.

<details><summary>💡 Hint 1 — what to think about</summary>

What belongs between your two clock readings? Only the stepping — not a model load, not the
reset. Which way up is the ratio: a small model on a laptop outruns reality, and the number
should say so. And is `sim_seconds` the time you were asked for, or the time your steps
actually bought?

</details>

<details><summary>💡 Hint 2 — the approach</summary>

Reset `data` first, so a second measurement does not start from the fallen pose the first
one left behind. Read `time.perf_counter()`, call your `step_for`, read it again. Build
`sim_seconds` from the step count it returned and the model's timestep, keep `steps` as the
integer it came back as, and divide simulated time by wall time.

</details>

In [ ]:
def real_time_factor(model, data, seconds: float) -> dict:
    """Measure how much faster than reality this machine simulates `seconds` of motion.

    In order: reset `data`, read `time.perf_counter()`, call your `step_for`, read the
    counter again.

    Example:
        >>> model, data = load_humanoid()
        >>> report = real_time_factor(model, data, 0.05)
        >>> sorted(report)
        ['real_time_factor', 'sim_seconds', 'steps', 'wall_seconds']
        >>> report["sim_seconds"] == report["steps"] * model.opt.timestep
        True

    Returns:
        dict with exactly these four keys:
          "steps"            int, from step_for
          "sim_seconds"      float, steps * model.opt.timestep
          "wall_seconds"     float, measured elapsed real time
          "real_time_factor" float, sim_seconds / wall_seconds
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_real_time_factor() -> None:
    model, data = load_humanoid()
    # 0.507 s is deliberately not a whole number of timesteps: on an exact multiple,
    # steps * timestep equals the seconds you were handed, and returning the argument
    # instead of the measured step count would slip through unnoticed.
    report = real_time_factor(model, data, 0.507)
    expected_keys = {"steps", "sim_seconds", "wall_seconds", "real_time_factor"}
    assert set(report) == expected_keys, (
        f"keys were {sorted(report)}, expected {sorted(expected_keys)} — return a dict with "
        "exactly those four names, spelled as in the docstring."
    )
    assert report["wall_seconds"] > 0.0, (
        "wall_seconds is not positive — you read perf_counter twice before stepping, or "
        "subtracted the two readings the wrong way round."
    )
    assert abs(report["sim_seconds"] - report["steps"] * model.opt.timestep) < 1e-9, (
        "sim_seconds does not equal steps * timestep — derive it from the step count you "
        "actually took, not from the `seconds` argument."
    )
    ratio = report["sim_seconds"] / report["wall_seconds"]
    assert abs(report["real_time_factor"] - ratio) < 1e-6, (
        f"real_time_factor is {report['real_time_factor']:.4f} but sim/wall is {ratio:.4f} — "
        "the ratio is simulated over wall; inverting it reports a fraction near zero."
    )
    assert report["real_time_factor"] > 1.0, (
        f"real_time_factor came out at {report['real_time_factor']:.4f} — a value that small "
        "means the ratio is upside down, or that you timed a reload instead of the stepping."
    )
    assert not isinstance(report["steps"], float), (
        f"steps came back as {type(report['steps']).__name__} — it counts mj_step calls, so "
        "it must be a whole number. math.ceil gives an int; numpy's np.ceil gives a float "
        "that still compares equal here and then breaks range() downstream."
    )
    # A second measurement on the same `data`: if the reset ran, the clock reads exactly what
    # this call's own steps bought, and nothing of the previous call survives.
    again = real_time_factor(model, data, 0.207)
    assert abs(data.time - again["steps"] * model.opt.timestep) < 1e-9, (
        f"a second call left data.time at {data.time}, not the "
        f"{again['steps'] * model.opt.timestep} its own {again['steps']} steps bought — the "
        "clock still carries the first run, so mujoco.mj_resetData never ran and every "
        "measurement after the first times an already-fallen pose."
    )
    print(f"exercise 3 looks right: {report['sim_seconds']:.3f} s simulated in "
          f"{report['wall_seconds']:.4f} s wall, RTF {report['real_time_factor']:.1f}x")

In [ ]:
_try("exercise 3", _check_real_time_factor, needs=("exercise 1",))

## 7. The payoff: a fall, measured

With all three instruments working, this cell watches the humanoid collapse under gravity
with no control at all, sampling the centre of mass on a fixed simulated-time grid. Every
number and every bar below is produced by the functions you just wrote.

In [ ]:
def fall_profile(samples: int = 10, interval: float = 0.2) -> list:
    """Return [(simulated time, COM height)] sampled every `interval` simulated seconds."""
    model, data = load_humanoid()
    mujoco.mj_forward(model, data)
    trace = [(data.time, com_height(model, data))]
    for _ in range(samples):
        step_for(model, data, interval)
        trace.append((data.time, com_height(model, data)))
    return trace


def _check_fall_profile() -> None:
    trace = fall_profile()
    start, end = trace[0][1], trace[-1][1]
    print(f"{'sim t (s)':>10s} {'COM z (m)':>10s}  profile")
    for t, z in trace:
        print(f"{t:10.2f} {z:10.4f}  {'#' * max(int(round(z * 40)), 0)}")
    assert end < start, (
        f"COM went from {start:.3f} to {end:.3f} — an uncontrolled humanoid under gravity "
        "must end lower than it started; check that step_for really calls mj_step."
    )
    print(f"\nfell {start - end:.3f} m of centre-of-mass height in {trace[-1][0]:.1f} s of "
          "simulated time, with zero control applied")


_try("the fall", _check_fall_profile, needs=("exercise 1", "exercise 2"))

## 8. Common mistakes

- **Assigning to `data.time`.** It moves the clock and simulates nothing. Only `mj_step`
  advances state; the clock is a consequence, not a control.
- **`int(seconds / timestep)`.** Truncation undershoots the request. 1.4 timesteps of work
  is 2 steps, and the last one overshoots — which is the honest answer.
- **Comparing `data.time` against `seconds` absolutely.** It works once, then reports zero
  steps forever, because the clock already exceeds the target. Measure from the time at
  which you were called.
- **Hard-coding the timestep.** This model overrides MuJoCo's documented default (sourced
  in `claims.yaml`). Read `model.opt.timestep` and your code survives the next model.
- **`data.xpos` for the centre of mass.** `xpos` is the body frame origin; `xipos` is where
  that body's mass actually sits. The cell below measures the gap on this model; it is not
  a rounding error.
- **Forgetting kinematics.** Writing `data.qpos` does not update `data.xipos`. Until
  `mj_forward` or `mj_step` runs, you are reading the previous pose.
- **`time.time` for benchmarking.** Use `time.perf_counter`: monotonic, higher resolution,
  and immune to a clock adjustment landing inside your measurement.

In [ ]:
# Watch the last two mistakes happen, measured rather than asserted. Nothing here is typed.
_stale = mujoco.MjData(MODEL)
mujoco.mj_forward(MODEL, _stale)
_frame_gap = np.abs(_stale.xpos[:, 2] - _stale.xipos[:, 2])
print(f"xpos vs xipos in z: worst body {_frame_gap.max():.4f} m, mean "
      f"{_frame_gap[_frame_gap > 0].mean():.4f} m over the "
      f"{int((_frame_gap > 0).sum())} bodies where the two differ at all")
_torso_before = float(_stale.xipos[1][2])
_stale.qpos[2] += 1.0                    # lift the robot a metre, then read derived state at once
print(f"wrote qpos[2] += 1.0 -> torso xipos z still reads {float(_stale.xipos[1][2]):.4f} "
      f"(it was {_torso_before:.4f})")
mujoco.mj_forward(MODEL, _stale)         # the one line that makes derived state true again
print(f"after mujoco.mj_forward it reads            {float(_stale.xipos[1][2]):.4f}  "
      f"(+{float(_stale.xipos[1][2]) - _torso_before:.4f} m, the lift you asked for)")
print(f"\nperf_counter monotonic={time.get_clock_info('perf_counter').monotonic}, "
      f"time() monotonic={time.get_clock_info('time').monotonic}"
      "  <- only one of these cannot go backwards mid-measurement")

## 9. Self-check

1. You need the mass of the left foot and the current velocity of the left knee. Where do
   they live?
   - (a) both in `mjModel`
   - (b) both in `mjData`
   - (c) mass in `mjModel`, velocity in `mjData`
   - (d) mass in `mjData`, velocity in `mjModel`

2. This model's `nq` exceeds its `nv` by exactly one. Why?
   - (a) one actuator is unactuated
   - (b) the free joint stores orientation as a 4-number quaternion but only a 3-number
         angular velocity
   - (c) MuJoCo reserves a slot in `qpos` for simulated time
   - (d) the world body contributes a position but no velocity

3. `real_time_factor` reports a value far above 1 on your laptop. What follows?
   - (a) the simulation is running too fast and is therefore inaccurate
   - (b) the integrator is silently skipping timesteps to keep up
   - (c) this model is cheap enough that a second of simulated motion costs far less than a
         second of your time; accuracy is a separate question, set by the timestep
   - (d) the timestep must be smaller than the documented default

4. You set `data.qpos[2] = 1.5` and immediately call `com_height(model, data)`. It returns
   the value from before the assignment. Why?
   - (a) `qpos` is read-only
   - (b) `com_height` caches its result
   - (c) `data.xipos` is a derived quantity, and nothing has recomputed it since you wrote
         to `qpos`
   - (d) the free joint ignores its z coordinate

Answers, with reasoning, are in this lesson's worked solution in the course repository. Mark yourself first
with the cell below, which tells you whether a letter is right without handing it to you.

In [ ]:
# Put your four letters here and run the cell. It marks them without revealing the answer:
# a wrong letter sends you back to the section that measured it, which is the point.
SELF_CHECK = {1: "?", 2: "?", 3: "?", 4: "?"}

_ANSWER_DIGESTS = {1: "d260b0fbba30524f", 2: "0c940df6b5fea42b",
                   3: "9230a39fdad2bf8e", 4: "f22b9958b9c74bec"}
_ANSWER_SECTIONS = {
    1: "section 1 — which of the two objects changed while the other one stayed put",
    2: "section 2 — the joint table you printed, and the free joint's 7-number qpos block",
    3: "section 6 — what your own real_time_factor measured, and what it says nothing about",
    4: "section 8 — the stale-xipos demonstration you just ran",
}


def _check_self_check(answers: dict = None) -> None:
    """Mark the four multiple-choice answers in SELF_CHECK, naming where to look again."""
    answers = SELF_CHECK if answers is None else answers
    wrong = []
    for q, digest in sorted(_ANSWER_DIGESTS.items()):
        got = str(answers.get(q, "?")).strip().lower()
        if hashlib.sha256(f"F15-L01-q{q}-{got}".encode()).hexdigest()[:16] != digest:
            wrong.append(q)
    for q in sorted(_ANSWER_DIGESTS):
        note = f"  -> re-read {_ANSWER_SECTIONS[q]}" if q in wrong else ""
        print(f"  q{q}: {'wrong' if q in wrong else 'right'}{note}")
    assert not wrong, (
        f"questions {wrong} are still wrong. Each one names the section that answers it "
        "above — go back to the cell you ran there rather than guessing another letter."
    )
    print("self-check: all four right")


def _check_self_check_answered() -> None:
    """The board's view of the self-check: letters left at "?" are not started, not wrong."""
    if all(str(v).strip() == "?" for v in SELF_CHECK.values()):
        raise NotImplementedError("not answered yet — put your four letters in SELF_CHECK "
                                  "above, then re-run this cell.")
    _check_self_check()


_try("self-check", _check_self_check_answered)

## What you built, and where it goes next

Three instruments — a clock you can trust, a posture sensor that reports the body rather
than the torso, and an honest stopwatch — plus the habit of reading every constant out of
`mjModel` instead of typing it. The rest of the Humanoid Lab flagship drives this same
model with actuators and a controller, and measures success with the centre-of-mass height
you just implemented.

In [ ]:
# Your progress board. Every check runs again here, quietly, so the board describes your code
# as it stands now: an exercise that was waiting on another you have since finished is marked
# afresh. In a script or under CI a failed check still ends the run non-zero; in a notebook it
# is a printed line.
_BOARD = [("exercise 1", _check_step_for, ()),
          ("exercise 2", _check_com_height, ()),
          ("exercise 3", _check_real_time_factor, ("exercise 1",)),
          ("self-check", _check_self_check_answered, ())]


def _progress_board() -> list:
    """Re-run every check without its output, print one line per exercise, return failures."""
    with contextlib.redirect_stdout(io.StringIO()):
        for label, check, needs in _BOARD:
            _try(label, check, needs)
    marks = {"passed": "✅", "failed": "❌", "not started": "⏳"}
    width = max(len(name) for name in _EXERCISES.values())
    print("progress board")
    for label, _, needs in _BOARD:
        state = _STATUS.get(label, "not started")
        waiting = [n for n in needs if _STATUS.get(n) != "passed"]
        note = ("  (waiting on " + " and ".join(waiting) + ")" if state == "not started"
                and waiting else "  (re-run its cell for the hint)" if state == "failed" else "")
        print(f"  {marks[state]} {label:<11s} {_EXERCISES[label]:<{width}s} {state}{note}")
    done = sum(_STATUS.get(label) == "passed" for label, _, _ in _BOARD)
    print(f"{done} of {len(_BOARD)} complete")
    return [label for label, state in _STATUS.items() if state == "failed"]


if __name__ == "__main__":
    _failed = _progress_board()
    if _failed:
        _message = "checks failed: " + ", ".join(_failed)
        if "ipykernel" in sys.modules:
            print(f"\n{_message} — each one printed its hint in its own cell above.")
        else:
            raise SystemExit(_message)